# HDB Resale Cleaning
This notebook combines the raw resale extracts, standardizes the core transaction fields, geocodes flat addresses, and produces the cleaned resale panel used by the downstream joins and model notebooks.


In [ ]:
import pandas as pd
from pathlib import Path
import numpy as np

RAW_DIR = Path("../data/raw")
RESALE_FILES = [
    RAW_DIR / "ResaleFlatPricesBasedonRegistrationDateFromMar2012toDec2014.csv",
    RAW_DIR / "ResaleFlatPricesBasedonRegistrationDateFromJan2015toDec2016.csv",
    RAW_DIR / "ResaleflatpricesbasedonregistrationdatefromJan2017onwards.csv",
]

dfs = [pd.read_csv(f) for f in RESALE_FILES]
df = pd.concat(dfs, ignore_index=True)
print(f"Total rows: {len(df):,}")
df.head()

## Load and Standardize Raw Transactions
The opening cells concatenate the yearly raw files, build a reusable address field for geocoding, and split the original transaction month into separate year and month columns.


In [ ]:
# Create address column by concatenating block and street_name (for OneMap API queries)
df["address"] = df["block"].astype(str) + " " + df["street_name"].astype(str)
df.drop(columns=["block", "street_name"], inplace=True)
df[["address"]].head()

In [ ]:
df.tail(10)

In [ ]:
# Split month (YYYY-MM) into year and month columns
df[["year", "month"]] = df["month"].str.split("-", expand=True)
df["year"] = df["year"].astype(int)
df["month"] = df["month"].astype(int)
df[["year", "month"]].head()

In [ ]:
df = df[df["year"] >= 2015]


In [ ]:
len(df["address"].unique())

## Geocode Flat Addresses
This section calls OneMap to recover latitude and longitude for each resale address. The API key is loaded from .env, and the resulting coordinates are merged back into the working resale dataframe.


## Getting LatLong for each flat from OneMap API
Create a `.env` file in the project root with your OneMap credentials:
```
EMAIL=your-onemap-email@example.com
PASSWORD=your-password
API_KEY= your-apikey (you can generate your api key if it expired under data_processing/onemapapi_key_generator.py)
```
Install `python-dotenv` if needed: `pip install python-dotenv`

In [ ]:
import os
import time
import requests
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv
from tqdm.auto import tqdm

# Load .env from project root (parent of data_processing/)
load_dotenv(Path.cwd().parent / ".env")
API_KEY = os.getenv("API_KEY")
if not API_KEY:
    raise ValueError("Set API_KEY in your .env file")

search_url = "https://www.onemap.gov.sg/api/common/elastic/search"
headers = {"Authorization": API_KEY}

def get_latlong_for_address(address, delay_sec=0.1):
    """Return (lat, lon) or (None, None) for a Singapore address."""
    if pd.isna(address) or not str(address).strip():
        return None, None
    params = {"searchVal": str(address).strip(), "returnGeom": "Y", "getAddrDetails": "Y"}
    try:
        r = requests.get(search_url, headers=headers, params=params)
        data = r.json()
        time.sleep(delay_sec)
        if data.get("results"):
            first = data["results"][0]
            return first.get("LATITUDE"), first.get("LONGITUDE")
    except Exception:
        print(f"Error getting lat/long for {address}")
        pass
    return None, None

# Geocode unique addresses, then merge back
unique_addresses = df["address"].unique()
geo = []
for addr in tqdm(unique_addresses, desc="Geocoding addresses"):
    lat, lon = get_latlong_for_address(addr)
    geo.append({"address": addr, "latitude": lat, "longitude": lon})

geo_df = pd.DataFrame(geo)
df = df.merge(geo_df, on="address", how="left")
df["geometry"] = df.apply(
    lambda r: f"POINT({r['latitude']} {r['longitude']})" if pd.notna(r.get("latitude")) and pd.notna(r.get("longitude")) else None,
    axis=1,
)
print(df[["address", "latitude", "longitude", "geometry"]].head(10))

In [ ]:
for col in df.columns:
    print(col)
    print(df[col].unique())
    print("\n")

# There is mixed integer and string formatting in the remaining_lease column (e.g. 70 vs "61 years 07 months"), so the following steps require parsing all values to a consistent total-months format.


## Normalize Lease Information
The remaining-lease field appears in mixed formats across source years, so this section converts everything into a common months-based measure and backfills missing values from the lease commencement year when needed.


In [ ]:
# Parse remaining_lease to total months (handles int years, "X years Y months", "X years", NaN)
import re

def parse_remaining_lease(val):
    if pd.isna(val):
        return None
    if isinstance(val, (int, float)):
        return int(val) * 12  # 2015-2016: numeric = years
    m = re.match(r"(\d+)\s*years?\s*(?:(\d+)\s*months?)?", str(val), re.I)
    if m:
        years, months = int(m.group(1)), int(m.group(2) or 0)
        return years * 12 + months
    return None

df["remaining_lease_months"] = df["remaining_lease"].apply(parse_remaining_lease)

# Fill NaN from lease_commence_date (99-yr lease; year-only so approximate)
mask = df["remaining_lease_months"].isna()
if mask.any():
    elapsed = df.loc[mask, "year"] - df.loc[mask, "lease_commence_date"] + (df.loc[mask, "month"] - 1) / 12
    df.loc[mask, "remaining_lease_months"] = (99 - elapsed).clip(lower=0).mul(12).round(0)

df["remaining_lease"] = df["remaining_lease_months"]
df.drop(columns=["remaining_lease_months"], inplace=True)
df[["remaining_lease"]].head()

In [ ]:
df.head()

## Save the Cleaned Resale Panel
After the core cleaning steps are complete, the notebook writes the consolidated transaction table to data/processed/cleaned_hdb_resale.csv for later enrichment.


In [ ]:
df.to_csv(Path("../data/processed/cleaned_hdb_resale.csv"), index=False)

## Exploratory Checks
The last cells are lightweight diagnostics for sanity-checking the cleaned panel, including basic time patterns and town-level counts.


## Data viz

In [ ]:
# Demographics over time (matplotlib)
import matplotlib.pyplot as plt

fig, axes = plt.subplots(3, 2, figsize=(14, 12))
fig.suptitle("HDB Resale Demographics Over Time", fontsize=14)

# 1. Town (top 8)
town_counts = df.groupby(["year", "town"]).size().unstack(fill_value=0)
top_towns = df["town"].value_counts().head(8).index
town_counts[top_towns].plot(ax=axes[0, 0], legend=True)
axes[0, 0].set_title("1. Town (top 8)")
axes[0, 0].set_xlabel("Year")

# 2. Flat type
ft = df.groupby(["year", "flat_type"]).size().unstack(fill_value=0)
ft.plot(ax=axes[0, 1], legend=True)
axes[0, 1].set_title("2. Flat Type")
axes[0, 1].set_xlabel("Year")

# 3 & 4. Storey range (same as story_range; top 8)
sr = df.groupby(["year", "storey_range"]).size().unstack(fill_value=0)
top_sr = df["storey_range"].value_counts().head(8).index
sr[top_sr].plot(ax=axes[1, 0], legend=True)
axes[1, 0].set_title("3. Storey Range (top 8)")
axes[1, 0].set_xlabel("Year")

# 5. Floor area sqm (mean per year)
fa = df.groupby("year")["floor_area_sqm"].mean()
axes[1, 1].plot(fa.index, fa.values, marker="o")
axes[1, 1].set_title("4. Floor Area (mean sqm)")
axes[1, 1].set_xlabel("Year")
axes[1, 1].set_ylabel("Mean sqm")

# 6. Flat model (top 8)
fm = df.groupby(["year", "flat_model"]).size().unstack(fill_value=0)
top_fm = df["flat_model"].value_counts().head(8).index
fm[top_fm].plot(ax=axes[2, 0], legend=True)
axes[2, 0].set_title("5. Flat Model (top 8)")
axes[2, 0].set_xlabel("Year")

# 7. Resale price (mean per year)
rp = df.groupby("year")["resale_price"].mean()
axes[2, 1].plot(rp.index, rp.values, marker="o", color="green")
axes[2, 1].set_title("6. Resale Price (mean)")
axes[2, 1].set_xlabel("Year")
axes[2, 1].set_ylabel("Mean price ($)")
axes[2, 1].ticklabel_format(style="plain", axis="y")

plt.tight_layout()
plt.show()

In [ ]:
# Town counts: clearer view (total sales per town + heatmap over time)
town_counts = df.groupby(["year", "town"]).size().unstack(fill_value=0)
town_totals = town_counts.sum().sort_values(ascending=True)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Left: horizontal bar chart (total sales per town)
town_totals.plot(kind="barh", ax=ax1, color="steelblue")
ax1.set_title("Total resale transactions by town")
ax1.set_xlabel("Count")
ax1.set_ylabel("Town")

# Right: heatmap (year x town) for trend over time
import numpy as np
im = ax2.imshow(town_counts.T, aspect="auto", cmap="YlOrRd")
ax2.set_xticks(range(len(town_counts.index)))
ax2.set_xticklabels(town_counts.index)
ax2.set_yticks(range(len(town_counts.columns)))
ax2.set_yticklabels(town_counts.columns, fontsize=8)
ax2.set_xlabel("Year")
ax2.set_title("Sales over time by town (heatmap)")
plt.colorbar(im, ax=ax2, label="Transactions")

plt.tight_layout()
plt.show()